# B-12: combined — B-10's ensemble + B-11's monthly rollout/downscale

**Hard sequencing dependency**: this notebook runs only after B-09/B-10/B-11 (D-53/D-54/D-55) have
results, and hardcodes their winning choices as frozen constants rather than re-running any grid:

- **From B-10**: alpha=1.0 (no AR blending -- blending with climatology never helped) and
  **unweighted ensemble** of RF+XGB+LightGBM+SARIMAX (the modest genuine win, mean R²=0.012).
- **From B-11**: the hybrid-calibration downscaling method (`recursive_rollout.
  downscale_monthly_to_daily`) — reuse a daily chain as the within-month shape, recenter to match
  an independently-derived monthly prediction.

**Structure**: (a) build B-11's monthly rollout for RF/XGB/LightGBM + SARIMAX (all at their
existing alpha=1.0 default — `monthly_rollout` has no blending option, so this literally
reproduces B-10's winning "no blend" config by construction); (b) unweighted ensemble of the four
monthly chains; (c) downscale using **B-10's own daily unweighted-ensemble chain** (recomputed
here) as the shape template — this is the piece that actually combines both ideas, since B-11 used
each *individual* model's own daily chain as its template, not an ensemble; (d) evaluate via
`bin_metrics`, compared directly against B-10's daily ensemble alone.

**Execution rule (user-confirmed)**: single-anchor (2021-12-16) smoke test here; the 5-anchor
sweep is a script extension (`b12_multi_anchor.py`), same precedent as B-09/B-10/B-11 — and here
it is not optional context, since it reverses this notebook's own single-anchor conclusion (§4).

In [1]:
from pathlib import Path
import sys, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
sys.path.insert(0, "../../src")

from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from statsmodels.tsa.statespace.sarimax import SARIMAX

import models.recursive_rollout as rr

HOURLY = Path("../../data/Hourly"); RESULTS = Path("../../results")
TOWER = 4
N_MONTHS = 13
DAILY_N_DAYS = 365
AR_COLS = ["ar_ch4_dlag1", "ar_ch4_dlag2", "ar_ch4_dlag3", "ar_ch4_dlag7", "ar_ch4_dlag14", "ar_ch4_drm7"]
AR_COLS_M = ["ar_ch4_mlag1", "ar_ch4_mlag2", "ar_ch4_mlag3"]
DUM = ["is_t2", "is_t4", "is_t9"]
EXOG_B = ["fx_lsu_dens", "fx_WS_mean", "fx_VPD_mean", "fx_USTAR_mean", "fx_PPFD_mean",
          "fx_DOY_sin", "fx_DOY_cos", "fx_is_growing"]
DAILY_ANCHOR = pd.Timestamp("2021-12-16")
MONTHLY_ANCHOR = pd.Timestamp("2021-11-01")

def fit_tree(algo, tr, feat_cols):
    imp = SimpleImputer(strategy="mean"); Xi = imp.fit_transform(tr[feat_cols].values)
    if algo == "RF":
        m = RandomForestRegressor(n_estimators=500, n_jobs=-1, random_state=42,
                                   min_samples_leaf=10, max_features=0.5)
    elif algo == "XGB":
        m = XGBRegressor(subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42,
                          max_depth=2, learning_rate=0.02, n_estimators=400, min_child_weight=10)
    elif algo == "LightGBM":
        m = LGBMRegressor(subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42,
                           num_leaves=7, min_child_samples=10, learning_rate=0.02, n_estimators=400,
                           verbosity=-1)
    else:
        raise ValueError(algo)
    m.fit(Xi, tr["target"].values); return m, imp

## 1  B-10's daily unweighted ensemble (recomputed here as the downscaling shape template)

In [2]:
dv = pd.read_csv(HOURLY/"forecast_daily_v2.csv", low_memory=False)
dv["Datetime"] = pd.to_datetime(dv["Datetime"], format="mixed")
FX_B = [c for c in dv.columns if c.startswith("fx")]
T = {t: dv[dv.tower == t].set_index("Datetime").sort_index() for t in [2, 4, 9]}
feat_cols_d = AR_COLS + FX_B + ["ar_fc_dlag1"] + DUM
daily_target_dates = pd.date_range(DAILY_ANCHOR + pd.Timedelta(days=1), periods=DAILY_N_DAYS, freq="D")
df4 = T[TOWER]
history_init_d = df4.loc[:DAILY_ANCHOR, "y_gapfilled"].copy()
fx_frame_d = df4.loc[daily_target_dates, FX_B + ["ar_fc_dlag1"]].copy()
fx_frame_d["is_t2"], fx_frame_d["is_t4"], fx_frame_d["is_t9"] = 0.0, 1.0, 0.0

pool = []
for t in [2, 4, 9]:
    df = T[t].copy(); df["target"] = df["y_gapfilled"]
    for d in DUM: df[d] = 1.0 if d == f"is_t{t}" else 0.0
    pool.append(df[df.index <= DAILY_ANCHOR])
tr_d = pd.concat(pool); tr_d = tr_d[tr_d["target"].notna()]

t0 = time.time()
daily_tree_chains = {}
for algo in ["RF", "XGB", "LightGBM"]:
    model, imp = fit_tree(algo, tr_d, feat_cols_d)
    daily_tree_chains[algo] = rr.tree_rollout(model, imp, feat_cols_d, fx_frame_d, history_init_d, DAILY_ANCHOR, n_days=DAILY_N_DAYS)

y = df4["y_gapfilled"].astype(float)
X = df4[EXOG_B].astype(float).ffill().bfill()
y_tr, X_tr = y.loc[:DAILY_ANCHOR], X.loc[:DAILY_ANCHOR]
best = None
for p in [1, 2]:
    for q in [0, 1]:
        try:
            m = SARIMAX(y_tr, exog=X_tr, order=(p, 1, q), enforce_stationarity=False, enforce_invertibility=False)
            res = m.fit(disp=False, maxiter=50)
            if best is None or res.aic < best[0]: best = (res.aic, (p, 1, q), res)
        except Exception:
            continue
order, res = best[1], best[2]
future_X = X.loc[daily_target_dates]
fc = res.get_forecast(steps=DAILY_N_DAYS, exog=future_X)
daily_sarimax_chain = pd.Series(fc.predicted_mean.values, index=daily_target_dates)

daily_ens_df = pd.DataFrame({**daily_tree_chains, "SARIMAX": daily_sarimax_chain})
daily_ens_unweighted = daily_ens_df.mean(axis=1)
print(f"daily ensemble built ({time.time()-t0:.0f}s), SARIMAX order={order}")

C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


daily ensemble built (23s), SARIMAX order=(2, 1, 1)


C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


## 2  B-11's monthly unweighted ensemble

In [3]:
dm = pd.read_csv(HOURLY/"forecast_monthly_v2.csv", low_memory=False)
dm["Datetime"] = pd.to_datetime(dm["Datetime"], format="mixed")
FX_M = [c for c in dm.columns if c.startswith("fx")]
M = {t: dm[dm.tower == t].set_index("Datetime").sort_index() for t in [2, 4, 9]}
feat_cols_m = AR_COLS_M + FX_M + DUM

pool = []
for t in [2, 4, 9]:
    df = M[t].copy(); df["target"] = df["y_gapfilled"]
    pool.append(df[df.index <= MONTHLY_ANCHOR])
tr_m = pd.concat(pool); tr_m = tr_m[tr_m["target"].notna()]

dm4 = M[TOWER]
history_init_m = dm4.loc[:MONTHLY_ANCHOR, "y_gapfilled"].copy()
target_months = pd.date_range(MONTHLY_ANCHOR + pd.DateOffset(months=1), periods=N_MONTHS, freq="MS")
fx_frame_m = dm4.loc[target_months, FX_M].copy()
fx_frame_m["is_t2"], fx_frame_m["is_t4"], fx_frame_m["is_t9"] = 0.0, 1.0, 0.0

t0 = time.time()
monthly_tree_chains = {}
for algo in ["RF", "XGB", "LightGBM"]:
    model, imp = fit_tree(algo, tr_m, feat_cols_m)
    monthly_tree_chains[algo] = rr.monthly_rollout(model, imp, feat_cols_m, fx_frame_m, history_init_m, MONTHLY_ANCHOR, n_months=N_MONTHS)

y = dm4["y_gapfilled"].astype(float)
X = dm4[EXOG_B].astype(float).ffill().bfill()
y_tr, X_tr = y.loc[:MONTHLY_ANCHOR], X.loc[:MONTHLY_ANCHOR]
best = None
for p in [1, 2]:
    for q in [0, 1]:
        try:
            m = SARIMAX(y_tr, exog=X_tr, order=(p, 1, q), enforce_stationarity=False, enforce_invertibility=False)
            res = m.fit(disp=False, maxiter=50)
            if best is None or res.aic < best[0]: best = (res.aic, (p, 1, q), res)
        except Exception:
            continue
future_X = X.loc[target_months]
fc = res.get_forecast(steps=N_MONTHS, exog=future_X)
monthly_sarimax_chain = pd.Series(fc.predicted_mean.values, index=target_months)

monthly_ens_df = pd.DataFrame({**monthly_tree_chains, "SARIMAX": monthly_sarimax_chain})
monthly_ens_unweighted = monthly_ens_df.mean(axis=1)
print(f"monthly ensemble built ({time.time()-t0:.0f}s)")

C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency 

monthly ensemble built (1s)


C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


## 3  Downscale monthly ensemble to daily, evaluate against B-10's daily ensemble

Verifies the recentering is exact by construction before trusting the result (same check as B-11).

In [4]:
daily_synth = rr.downscale_monthly_to_daily(monthly_ens_unweighted, daily_ens_unweighted.reindex(daily_target_dates))

check = daily_synth.groupby(daily_synth.index.to_period("M")).mean()
monthly_by_period = monthly_ens_unweighted.copy(); monthly_by_period.index = monthly_by_period.index.to_period("M")
max_diff = (check - monthly_by_period.reindex(check.index)).abs().max()
print(f"downscale recenter exactness check: max abs diff = {max_diff:.10f} (expect 0)")

b09_chains = pd.read_csv(RESULTS/"b09_chains.csv", index_col=0, parse_dates=True)
y_true_daily = b09_chains["y_true"].reindex(daily_target_dates).values
anchor_val_daily = df4.loc[DAILY_ANCHOR, "y_gapfilled"]
persist_daily = rr.chain_persistence(anchor_val_daily, DAILY_N_DAYS)

yp_b12 = daily_synth.reindex(daily_target_dates).values
bm_b12 = rr.bin_metrics(y_true_daily, yp_b12, daily_target_dates, DAILY_ANCHOR, y_persist=persist_daily)
bm_b12["model"] = "B12_monthly_ensemble_downscaled"

yp_b10 = daily_ens_unweighted.reindex(daily_target_dates).values
bm_b10 = rr.bin_metrics(y_true_daily, yp_b10, daily_target_dates, DAILY_ANCHOR, y_persist=persist_daily)
bm_b10["model"] = "B10_daily_ensemble_original"

R = pd.concat([bm_b12, bm_b10], ignore_index=True)
R.to_csv(RESULTS/"b12_summary.csv", index=False)

pd.set_option("display.width", 200)
print("\n=== R2 by model x bin, single anchor 2021-12-16 ===")
print(R.pivot_table(index="model", columns="bin", values="R2").round(3).to_string())
print("\n=== MASE by model x bin ===")
print(R.pivot_table(index="model", columns="bin", values="MASE").round(3).to_string())

downscale recenter exactness check: max abs diff = 0.0000000000 (expect 0)

=== R2 by model x bin, single anchor 2021-12-16 ===
bin                                1-7  181-270  271-365  31-90   8-30  91-180
model                                                                         
B10_daily_ensemble_original     -2.396    0.293   -0.436 -0.035 -0.200   0.168
B12_monthly_ensemble_downscaled -2.386    0.302    0.066 -0.154 -0.052   0.098

=== MASE by model x bin ===
bin                                1-7  181-270  271-365  31-90   8-30  91-180
model                                                                         
B10_daily_ensemble_original      1.532    0.967    1.491  1.159  1.156   0.859
B12_monthly_ensemble_downscaled  1.531    0.881    1.106  1.226  1.065   0.861


## 4  Multi-anchor (2018-2022) extension — reverses this notebook's own single-anchor result

At this single 2021 anchor, B-12 looks like a clear improvement over B-10's daily ensemble alone
(overall n-weighted R²: 0.082 vs -0.034). Per this project's own repeated lesson (B-09 §3, B-10 §3,
now this), that single-anchor read is not trustworthy — extended to the same 5-anchor sweep as
script `b12_multi_anchor.py` (not re-executed here). Results: `results/b12_multi_anchor.csv`.

**The multi-anchor result reverses the single-anchor read**: mean R²/MASE across 5 anchors —
`B10_daily_ensemble_original` 0.012/0.975, `B12_monthly_ensemble_downscaled` **-0.011/0.993** —
B-12 is now slightly *worse* than B-10 alone, not better. By bin, the short-lead bins improve
(1-7: -3.292->-1.986; 8-30: -1.006->-0.740) and 271-365 improves slightly (0.011->0.072), but
181-270 and 91-180 get worse (0.232->0.190; 0.001->-0.152) -- a net wash to slightly negative once
n-weighted across all bins and all 5 anchors. This is exactly the same mechanism B-11 (D-55)
already found: the ensemble's own daily shape errors are inherited unchanged by the downscaling,
and averaging across 5 different years' worth of that same effect erodes the single-anchor gain
this notebook's own §3 seemed to show.

**This is itself a third demonstration of this project's own headline lesson**: don't trust a
single-anchor backtest, even a seemingly clean one. **Combining B-10's ensemble win with B-11's
monthly-downscaling framework does not produce a further improvement** — B-10's ensemble alone
remains the best available daily-resolution recursive-rollout result (mean R²=0.012).